In [1]:
import cvxpy as cp
import numpy as np
import pandas as pd
import scs
from isometry_pursuit.transformation import exponential_transformation
import numpy as np
from isometry_pursuit.experiment import run_tensor_experiment

data_tensor = np.load('/Users/samsonkoelle/isometry-pursuit/data/ethanol_tensor.npy')
print(data_tensor.shape)

r = 1  # or whichever replicate
X = data_tensor[:, :, r]
X_t = exponential_transformation(X)

D, P = X_t.shape
beta_var = cp.Variable((P, D))

prob = cp.Problem(
    cp.Minimize(cp.sum(cp.norm(beta_var, axis=1))),
    [X_t @ beta_var == np.eye(D)]
)

# Canonicalize once through CVXPY
data, chain, inverse_data = prob.get_problem_data(cp.SCS)

# Call SCS directly, with per-iteration CSV logging
solver = scs.SCS(
    data,
    cone=data["dims"],
    eps_abs=1e-4,
    eps_rel=1e-4,
    max_iters=100000,
    verbose=False,
    log_csv_filename="scs_trace.csv",
)

sol = solver.solve()

# Push solution back into the CVXPY problem if you want
prob.unpack_results(sol, chain, inverse_data)

trace = pd.read_csv("scs_trace.csv")
print(trace.head())
print(sol["info"])

(2, 756, 5)


ValueError: Error parsing inputs
